In [ ]:
# Dataset: https://huggingface.co/datasets/HuggingFaceFW/fineweb
# tokeniser: https://tiktokenizer.vercel.app/?model=gpt-3.5-turbo
# gtp2: tokeniser: https://github.com/openai/gpt-2/blob/master/src/encoder.py

In [156]:
text = "The Tokenizer is a necessary and pervasive component of Large Language Models (LLMs), where it translates between strings and tokens (text chunks). Tokenizers are a completely separate stage of the LLM pipeline: they have their own training sets, training algorithms (Byte Pair Encoding), and after training implement two fundamental functions: encode() from strings to tokens, and decode() back from tokens to strings. In this lecture we build from scratch the Tokenizer used in the GPT series from OpenAI. In the process, we will see that a lot of weird behaviors and problems of LLMs actually trace back to tokenization. We'll go through a number of these issues, discuss why tokenization is at fault, and why someone out there ideally finds a way to delete this stage entirely."

In [157]:
tokens = text.encode("utf-8")
tokens = list(map(int, tokens))
print(len(tokens))

781


In [158]:
def common_pairs(tokens):
    token_map = {}
    for i in range(len(tokens)-1):
        joined_token = (str(tokens[i]),str(tokens[i+1]))
        token_map[joined_token] = token_map.get(joined_token, 0)+1
    
    return sorted(((v,k) for k,v in token_map.items()), reverse=True)

In [159]:
chr(32), chr(116)

(' ', 't')

In [160]:
# Map the common pairs to new token values
def new_token_mapping_function(paired_tokens, new_token_id, new_token_mapping):
    
    for i in paired_tokens:
        frequency = i[0]
        pair = i[1]
        if(frequency < 5):
            break
        else:
            new_token_mapping[pair] = new_token_id
            new_token_id += 1

    return new_token_mapping

In [161]:
# Replace the pairs with new tokens in the data

def encode_token(tokens, new_token_mapping):
    encoded_tokens = []
    i = 0
    while(i < len(tokens)-1):
        joined_token = (str(tokens[i]),str(tokens[i+1]))
        if (joined_token in new_token_mapping):
            encoded_tokens.append(new_token_mapping[joined_token])
            i += 1
        else:
            encoded_tokens.append(tokens[i])
        i += 1

    return encoded_tokens

In [162]:
new_token_id = 256
new_token_mapping = {}
paired_tokens = common_pairs(tokens)
new_token_mapping = new_token_mapping_function(paired_tokens, new_token_id, new_token_mapping)
encoded_tokens = encode_token(tokens, new_token_mapping)

781
Compression ratio:  {0.677336747759283}


In [163]:
print(f"Compression ratio: ", {len(tokens)/len(encoded_tokens)})

Compression ratio:  {1.4763705103969755}


In [166]:
# print(new_token_mapping)

def invert_token_mapping(new_token_mapping):
    inverted_token_map = {}
    for k,v in new_token_mapping.items():
        inverted_token_map[v] = k
    return inverted_token_map

inverted_token_map = invert_token_mapping(new_token_mapping)
        

In [170]:
def decode_token(tokens, inverted_token_map):
    decoded_tokens = []
    i = 0
    while(i < len(tokens)-1):
        if(int(tokens[i]) >= 256):
            decoded_tokens.append(inverted_token_map[int(tokens[i])][0])
            decoded_tokens.append(inverted_token_map[int(tokens[i])][1])
        else:
            decoded_tokens.append(tokens[i])
        i += 1

    return decoded_tokens

print(len(decode_token(encoded_tokens, inverted_token_map)))

779


In [ ]:
# The gpt2 tokenizer uses regular expressions to avoid seperate token creations for word + white_spaces, word + punctuations etc
# Instead of creating the tokens array over the entire data, each sentence is split using the regex, and the byte encoding algorithm is applied over each split word.
"Hello's, world! I'm 42."
pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""